# model violin graph Fig. 5

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy import stats
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

SCI_STYLE = {
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 900,
    'savefig.format': 'tiff',
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
}

plt.rcParams.update(SCI_STYLE)

df = pd.read_csv('../final_report1.csv')

custom_model_order = ['JointSyn', 'CGMS', 'DeepDDS', 'MTLSynergy', 'MLP', 'XGB', 'RF']

target_mapping = {
    'synergy_loewe': 'Loewe',
    'synergy_hsa': 'HSA',
    'synergy_bliss': 'Bliss',
    'synergy_zip': 'ZIP'
}

metrics_config = {
    'MSE': {
        'metric_col': 'MSE_Mean',
        'title_suffix': 'MSE (Lower is better)',
        'ranking_direction': 'ascending'
    },
    'PCC': {
        'metric_col': 'PCC_Mean',
        'title_suffix': 'PCC (Higher is better)',
        'ranking_direction': 'descending'
    },
    'R2': {
        'metric_col': 'R2_Mean',
        'title_suffix': 'R-squared (Higher is better)',
        'ranking_direction': 'descending'
    },
    'RMSE': {
        'metric_col': 'RMSE_Mean',
        'title_suffix': 'RMSE (Lower is better)',
        'ranking_direction': 'ascending'
    },
    'MAE': {
        'metric_col': 'MAE_Mean',
        'title_suffix': 'MAE (Lower is better)',
        'ranking_direction': 'ascending'
    }
}

targets = sorted(df['Target'].unique())

model_colors = {
    'JointSyn': '#4C72B0',  
    'CGMS': '#55A868',   
    'DeepDDS': '#C44E52', 
    'MTLSynergy': '#DDAA33', 
    'MLP': '#8172B2',    
    'XGB': '#CCB974',    
    'RF': '#FF7F00'      
}

for metric, config in metrics_config.items():
    metric_df = df.copy()
    
    def rank_within_study_target(group):
        metric_col = config['metric_col']
        if config['ranking_direction'] == 'ascending':
            group['rank'] = group[metric_col].rank(method='min', ascending=True)
        else:
            group['rank'] = group[metric_col].rank(method='min', ascending=False)
        return group

    ranked_df = metric_df.groupby(['Study', 'Target']).apply(rank_within_study_target).reset_index(drop=True)

    fig, axs = plt.subplots(2, 2, figsize=(3.5, 4.5), sharex=True, sharey=True, 
                           gridspec_kw={'wspace': 0.05, 'hspace': 0.15})  

    axs[0,0].invert_yaxis()

    for i, target in enumerate(targets):
        row_idx = i // 2
        col_idx = i % 2
        ax = axs[row_idx, col_idx]

        target_df = ranked_df[ranked_df['Target'] == target]

        plot_data = []
        positions = []
        labels = custom_model_order 

        for j, model in enumerate(custom_model_order):
            model_ranks = target_df[target_df['Model'] == model]['rank'].values
            plot_data.append(model_ranks)
            positions.append(j+1)  

        vp = ax.violinplot(
            plot_data,
            positions=positions,
            widths=0.7,  
            showmeans=False,
            showmedians=False,
            showextrema=False
        )

        for pc, model in zip(vp['bodies'], custom_model_order):
            pc.set_facecolor(model_colors[model])
            pc.set_edgecolor('black')
            pc.set_alpha(0.7)
            pc.set_linewidth(0.5)  

        bp = ax.boxplot(
            plot_data,
            positions=positions,
            widths=0.12,  
            patch_artist=True,
            boxprops=dict(facecolor='white', alpha=0.9, linewidth=0.8),
            medianprops=dict(color='black', linewidth=1),
            whiskerprops=dict(linewidth=0.8),
            capprops=dict(linewidth=0.8),
            showfliers=False
        )

        mapped_target = target_mapping.get(target, target)
        ax.set_title(f'{mapped_target}', fontsize=8, pad=6)

        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)

        ax.grid(axis='y', alpha=0.2, linestyle='--', linewidth=0.5)
        ax.set_axisbelow(True)

        model_data = {}
        for j, model in enumerate(custom_model_order):
            model_data[model] = {
                'data': target_df[target_df['Model'] == model]['rank'].values,
                'position': positions[j]
            }

        min_rank = min([min(ranks) for ranks in plot_data]) 
        max_rank = max([max(ranks) for ranks in plot_data]) 

        base_height = min_rank - 0.3

        comparisons = list(itertools.combinations(custom_model_order, 2))
        comparisons.sort(key=lambda pair: abs(model_data[pair[0]]['position'] - model_data[pair[1]]['position']))

        height_levels = {}
        max_significance_height = base_height  
        
        for comp in comparisons:
            model1, model2 = comp
            pos1 = model_data[model1]['position']
            pos2 = model_data[model2]['position']
            
            center = (pos1 + pos2) / 2

            heights = [h for (s, e), h in height_levels.items()
                      if not (e < pos1 or s > pos2)]
            if heights:
                new_height = max(heights) + 0.2
            else:
                new_height = base_height
                
            height_levels[(pos1, pos2)] = new_height

            if new_height < max_significance_height:
                max_significance_height = new_height

            data1 = model_data[model1]['data']
            data2 = model_data[model2]['data']

            if len(data1) < 2 or len(data2) < 2:
                continue

            u_stat, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')

            if p_value < 0.001:
                stars = '***'
            elif p_value < 0.01:
                stars = '**'
            elif p_value < 0.05:
                stars = '*'
            else:
                continue

            ax.plot([pos1, pos2], [new_height, new_height], 
                    color='gray', linewidth=0.8, zorder=1)

            ax.text(
                center, 
                new_height - 0.15,
                stars, 
                ha='center', 
                va='top', 
                fontsize=7,
                fontweight='bold',
                color='black'
            )

        ax.set_ylim(max_rank + 0.5, max_significance_height - 0.5)

    fig.text(0.02, 0.5, 'Rank (1 = Best)', va='center', rotation='vertical', fontsize=8)

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5) 

    plt.savefig(f'model_performance_violin_{metric}.tiff', dpi=900, bbox_inches='tight', facecolor='white')
    plt.close(fig)


C:\Users\84991\AppData\Local\Temp\ipykernel_36852\3270889195.py:262: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)  # 减小内边距
C:\Users\84991\AppData\Local\Temp\ipykernel_36852\3270889195.py:262: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)  # 减小内边距
C:\Users\84991\AppData\Local\Temp\ipykernel_36852\3270889195.py:262: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)  # 减小内边距
C:\Users\84991\AppData\Local\Temp\ipykernel_36852\3270889195.py:262: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)  # 减小内边距


所有指标的小提琴图已按SCI要求生成
